In [21]:
# Step 1: Load the dataset
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

DATA_PATH = Path("DataCoSupplyChainDataset.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "DataCoSupplyChainDataset.csv was not found. Place it in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH, encoding="latin-1")
raw_df = df.copy(deep=True)  # reference copy; all checks below are read-only

print("Dataset loaded successfully")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

Dataset loaded successfully
Rows: 180,519
Columns: 53


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [22]:
# Step 2: Dataset overview and exact duplicate baseline
print("Total columns:", len(df.columns))
print("Exact duplicate rows:", int(df.duplicated().sum()))
display(pd.DataFrame({"Column": df.columns, "Data Type": df.dtypes.astype(str).values,
                      "Non-null Values": df.notna().sum().values}))


Total columns: 53
Exact duplicate rows: 0


,Column,Data Type,Non-null Values
0,Type,object,180519
1,Days for shipping (real),int64,180519
2,Days for shipment (scheduled),int64,180519
3,Benefit per order,float64,180519
4,Sales per customer,float64,180519
5,Delivery Status,object,180519
6,Late_delivery_risk,int64,180519
7,Category Id,int64,180519
8,Category Name,object,180519
9,Customer City,object,180519


In [23]:
# Step 3: Configure validation checks

numeric_rules = {
    "Order Item Quantity": ("Quantity should be greater than zero", lambda s: s <= 0),
    "Product Price": ("Product price should not be negative", lambda s: s < 0),
    "Order Item Product Price": ("Order item product price should not be negative", lambda s: s < 0),
    "Order Item Discount Rate": ("Discount rate should be between 0 and 1 as a broad validation bound", lambda s: (s < 0) | (s > 1)),
    "Late_delivery_risk": ("Risk flag should be binary (0 or 1)", lambda s: ~s.isin([0, 1])),
    "Days for shipping (real)": ("Actual shipping duration should not be negative", lambda s: s < 0),
    "Days for shipment (scheduled)": ("Scheduled shipping duration should not be negative", lambda s: s < 0),
    "Latitude": ("Latitude must be between -90 and 90", lambda s: (s < -90) | (s > 90)),
    "Longitude": ("Longitude must be between -180 and 180", lambda s: (s < -180) | (s > 180)),
    "Order Item Discount": ("Discount amount should not be negative", lambda s: s < 0),
    "Order Item Profit Ratio": ("Flag only extreme ratio values outside a broad review range of -10 to 10", lambda s: (s < -10) | (s > 10)),
}

date_columns = ["order date (DateOrders)", "shipping date (DateOrders)"]
category_columns = [
    "Type", "Delivery Status", "Customer Segment", "Market", "Order Region",
    "Order Status", "Shipping Mode", "Category Name", "Department Name",
    "Customer State", "Order Country", "Product Name"
]
mapping_pairs = [
    ("Category Id", "Category Name"),
    ("Department Id", "Department Name"),
    ("Product Card Id", "Product Name"),
    ("Product Category Id", "Category Name"),
]
postal_columns = ["Customer Zipcode", "Order Zipcode"]

# Fail early if the source file schema differs from the expected dataset.
required = list(numeric_rules) + date_columns + category_columns + postal_columns
required += [c for pair in mapping_pairs for c in pair]
missing_columns = sorted(set(required) - set(df.columns))
if missing_columns:
    raise KeyError(f"Expected columns missing from dataset: {missing_columns}")

print("Configured checks:")
print(f"Numeric: {len(numeric_rules)} | Date parse: {len(date_columns)} | Chronology: 1 | "
      f"Category consistency: {len(category_columns)} | ID-label mappings: {len(mapping_pairs)*2} | "
      f"Postal data types: {len(postal_columns)}")
print("Total:", len(numeric_rules) + len(date_columns) + 1 + len(category_columns) + len(mapping_pairs)*2 + len(postal_columns))


Configured checks:
Numeric: 11 | Date parse: 2 | Chronology: 1 | Category consistency: 12 | ID-label mappings: 8 | Postal data types: 2
Total: 36


In [24]:
# Step 4: Run all 36 checks and build a consistent validation table
validation_rows = []
category_variant_rows = []
mapping_detail_rows = []
findings_rows = []

# 4A. Numeric rules (11 checks)
for column, (rule_text, rule_fn) in numeric_rules.items():
    numeric_values = pd.to_numeric(df[column], errors="coerce")
    nonempty = df[column].notna() & df[column].astype(str).str.strip().ne("")
    parse_failures = int((nonempty & numeric_values.isna()).sum())
    invalid_mask = rule_fn(numeric_values).fillna(False)
    affected = int(invalid_mask.sum())
    validation_rows.append({
        "Validation Area": "Numeric", "Column(s)": column, "Validation Rule": rule_text,
        "Result": "No violations detected" if affected == 0 else "Violations detected",
        "Affected Rows / Count": affected, "Status": "Pass" if affected == 0 else "Review",
        "Notes": f"Numeric parse failures: {parse_failures}" if parse_failures else None
    })
    if affected:
        findings_rows.append({"Validation Area": "Numeric", "Column Name": column,
            "Issue / Observation": rule_text, "Affected Rows": affected,
            "Example Values / Evidence": df.loc[invalid_mask, column].head(5).astype(str).tolist(),
            "Suggested Action": "Review source values and confirm the business rule before correction.",
            "Finding Status": "Needs review"})
    if parse_failures:
        findings_rows.append({"Validation Area": "Numeric data type", "Column Name": column,
            "Issue / Observation": "Non-empty values could not be parsed as numeric",
            "Affected Rows": parse_failures,
            "Example Values / Evidence": df.loc[nonempty & numeric_values.isna(), column].head(5).astype(str).tolist(),
            "Suggested Action": "Inspect source values; do not silently coerce or overwrite the raw dataset.",
            "Finding Status": "Needs review"})

# 4B. Date parsing (2 checks)
parsed_dates = {}
for column in date_columns:
    original = df[column]
    parsed = pd.to_datetime(original, errors="coerce")
    nonempty = original.notna() & original.astype(str).str.strip().ne("")
    invalid_mask = nonempty & parsed.isna()
    invalid_count = int(invalid_mask.sum())
    blank_count = int((~nonempty).sum())
    parsed_dates[column] = parsed
    validation_rows.append({
        "Validation Area": "Date", "Column(s)": column,
        "Validation Rule": "Value must be parseable as a date/time",
        "Result": "All non-empty values parsed successfully" if invalid_count == 0 else "Unparseable dates found",
        "Affected Rows / Count": invalid_count, "Status": "Pass" if invalid_count == 0 else "Review",
        "Notes": f"Blank values: {blank_count}"
    })
    if invalid_count:
        findings_rows.append({"Validation Area": "Date", "Column Name": column,
            "Issue / Observation": "Non-empty date/time value could not be parsed",
            "Affected Rows": invalid_count,
            "Example Values / Evidence": df.loc[invalid_mask, column].head(5).astype(str).tolist(),
            "Suggested Action": "Review the original date strings and standardize only in a working copy.",
            "Finding Status": "Needs review"})

# 4C. Date chronology (1 check)
order_col, ship_col = date_columns
comparable = parsed_dates[order_col].notna() & parsed_dates[ship_col].notna()
chronology_mask = comparable & (parsed_dates[ship_col] < parsed_dates[order_col])
chronology_count = int(chronology_mask.sum())
validation_rows.append({
    "Validation Area": "Date chronology",
    "Column(s)": f"{order_col} / {ship_col}",
    "Validation Rule": "Shipping date must not precede order date",
    "Result": "No shipping-before-order records found" if chronology_count == 0 else "Shipping-before-order records found",
    "Affected Rows / Count": chronology_count,
    "Status": "Pass" if chronology_count == 0 else "Review",
    "Notes": None
})
if chronology_count:
    findings_rows.append({"Validation Area": "Date chronology", "Column Name": f"{order_col} / {ship_col}",
        "Issue / Observation": "Shipping date is earlier than order date", "Affected Rows": chronology_count,
        "Example Values / Evidence": df.loc[chronology_mask, [order_col, ship_col]].head(5).astype(str).to_dict("records"),
        "Suggested Action": "Investigate flagged records; do not auto-correct dates.", "Finding Status": "Needs review"})

# 4D. Category consistency (12 checks). Trim and casefold are used for comparison only;
# the source values are not modified. Count variation groups separately from affected rows.
for column in category_columns:
    original = df[column].dropna().astype(str)
    trimmed = original.str.strip()
    normalized = trimmed.str.casefold()
    pairs = pd.DataFrame({"Original": original, "Trimmed": trimmed, "Normalized": normalized})
    variation_counts = pairs.groupby("Normalized")["Trimmed"].nunique()
    variant_groups = variation_counts[variation_counts > 1]
    group_count = int(len(variant_groups))
    affected_rows = int(normalized.isin(variant_groups.index).sum()) if group_count else 0
    for normalized_label in variant_groups.index:
        raw_variants = sorted(pairs.loc[pairs["Normalized"] == normalized_label, "Original"].unique().tolist())
        category_variant_rows.append({"Column Name": column, "Normalized Label": normalized_label,
            "Raw Variants": " | ".join(raw_variants), "Distinct Raw Variants": len(raw_variants),
            "Affected Rows": int((normalized == normalized_label).sum())})
    validation_rows.append({
        "Validation Area": "Category consistency", "Column(s)": column,
        "Validation Rule": "Check for spelling/case variants after trimming whitespace",
        "Result": "No additional spelling/case variants found" if group_count == 0 else f"{group_count} variation group(s) found",
        "Affected Rows / Count": affected_rows,
        "Status": "Pass" if group_count == 0 else "Review",
        "Notes": f"Variation groups: {group_count}. Whitespace-only findings were documented in the earlier profiling report."
    })
    if group_count:
        findings_rows.append({"Validation Area": "Category consistency", "Column Name": column,
            "Issue / Observation": f"{group_count} normalized label group(s) contain spelling/case variants",
            "Affected Rows": affected_rows,
            "Example Values / Evidence": [r for r in category_variant_rows if r["Column Name"] == column][:5],
            "Suggested Action": "Review category values with the data dictionary before mapping labels.",
            "Finding Status": "Needs review"})

# 4E. ID-label mapping (8 checks). Distinguish conflicting groups from affected records.
for id_col, label_col in mapping_pairs:
    mapping = df[[id_col, label_col]].dropna().copy()
    mapping[id_col] = mapping[id_col].astype(str).str.strip()
    mapping[label_col] = mapping[label_col].astype(str).str.strip().str.casefold()
    # Each ID should map to one label.
    id_label_counts = mapping.groupby(id_col)[label_col].nunique()
    bad_ids = id_label_counts[id_label_counts > 1].index
    id_rows = int(mapping[id_col].isin(bad_ids).sum())
    mapping_detail_rows.append({"ID Column": id_col, "Label Column": label_col,
        "Mapping Direction": "One ID → multiple labels", "ID / Label": " | ".join(map(str, bad_ids[:10])) if len(bad_ids) else "",
        "Observed Values": "Conflicting labels for listed IDs" if len(bad_ids) else "No conflicts",
        "Conflict Groups": int(len(bad_ids)), "Affected Rows": id_rows})
    validation_rows.append({"Validation Area": "ID-label mapping", "Column(s)": f"{id_col} → {label_col}",
        "Validation Rule": "Each ID should map consistently to one label",
        "Result": "No ID-to-label conflicts found" if len(bad_ids) == 0 else f"{len(bad_ids)} ID conflict group(s) found",
        "Affected Rows / Count": id_rows, "Status": "Pass" if len(bad_ids) == 0 else "Review", "Notes": None})
    if len(bad_ids):
        findings_rows.append({"Validation Area": "ID-label mapping", "Column Name": f"{id_col} / {label_col}",
            "Issue / Observation": "One ID maps to multiple labels", "Affected Rows": id_rows,
            "Example Values / Evidence": mapping[mapping[id_col].isin(bad_ids)].head(5).to_dict("records"),
            "Suggested Action": "Verify against the official mapping; do not assume it is an error.", "Finding Status": "Needs review"})

    # Each normalized label should map to one ID. This is a review signal, not necessarily an error.
    label_id_counts = mapping.groupby(label_col)[id_col].nunique()
    bad_labels = label_id_counts[label_id_counts > 1].index
    label_rows = int(mapping[label_col].isin(bad_labels).sum())
    for label in bad_labels:
        ids = sorted(mapping.loc[mapping[label_col] == label, id_col].unique().tolist())
        mapping_detail_rows.append({"ID Column": id_col, "Label Column": label_col,
            "Mapping Direction": "One label → multiple IDs", "ID / Label": label,
            "Observed Values": " | ".join(ids), "Conflict Groups": 1, "Affected Rows": int((mapping[label_col] == label).sum())})
    validation_rows.append({"Validation Area": "ID-label mapping", "Column(s)": f"{label_col} ← {id_col}",
        "Validation Rule": "Each label should map consistently to one ID",
        "Result": "No label-to-ID conflicts found" if len(bad_labels) == 0 else f"{len(bad_labels)} label group(s) associated with multiple IDs",
        "Affected Rows / Count": label_rows, "Status": "Pass" if len(bad_labels) == 0 else "Review",
        "Notes": "A label may legitimately be reused across IDs; verify with the official data dictionary." if len(bad_labels) else None})
    if len(bad_labels):
        findings_rows.append({"Validation Area": "ID-label mapping", "Column Name": f"{id_col} / {label_col}",
            "Issue / Observation": "One label is associated with multiple IDs; review whether expected",
            "Affected Rows": label_rows,
            "Example Values / Evidence": [{"Label": label, "IDs": sorted(mapping.loc[mapping[label_col] == label, id_col].unique().tolist())} for label in list(bad_labels[:5])],
            "Suggested Action": "Validate against the official category/product mapping; do not assume it is an error.",
            "Finding Status": "Needs review"})

# 4F. Postal code data types (2 checks). A type review is not counted as invalid values.
for column in postal_columns:
    nonempty = df[column].notna() & df[column].astype(str).str.strip().ne("")
    values = pd.to_numeric(df.loc[nonempty, column], errors="coerce")
    non_integer_mask = values.notna() & ((values % 1) != 0)
    non_integer_count = int(non_integer_mask.sum())
    current_dtype = str(df[column].dtype)
    type_needs_review = not (pd.api.types.is_string_dtype(df[column].dtype) or pd.api.types.is_object_dtype(df[column].dtype))
    validation_rows.append({
        "Validation Area": "Data type", "Column(s)": column,
        "Validation Rule": "Postal codes should be treated as identifiers/text, not quantities",
        "Result": f"{non_integer_count} non-integer-like non-empty values; current dtype: {current_dtype}",
        "Affected Rows / Count": non_integer_count,
        "Status": "Review recommended" if type_needs_review else ("Pass" if non_integer_count == 0 else "Review"),
        "Notes": "If numeric, convert only in a cleaned copy after checking source values; leading zeros cannot be recovered if already lost."
    })
    if type_needs_review or non_integer_count:
        examples = df.loc[nonempty, column].head(5).astype(str).tolist()
        findings_rows.append({"Validation Area": "Data type", "Column Name": column,
            "Issue / Observation": "Postal code is not stored as identifier text" if type_needs_review else "Non-integer-like postal code values found",
            "Affected Rows": int(nonempty.sum()) if type_needs_review else non_integer_count,
            "Example Values / Evidence": f"dtype={current_dtype}; examples={examples}",
            "Suggested Action": "In a cleaned copy, convert verified integer-like values to text carefully; preserve leading zeros where source data supports them.",
            "Finding Status": "Standardization opportunity" if type_needs_review and non_integer_count == 0 else "Needs review"})

validation_checks = pd.DataFrame(validation_rows)
category_variants = pd.DataFrame(category_variant_rows, columns=["Column Name", "Normalized Label", "Raw Variants", "Distinct Raw Variants", "Affected Rows"])
id_label_mapping = pd.DataFrame(mapping_detail_rows, columns=["ID Column", "Label Column", "Mapping Direction", "ID / Label", "Observed Values", "Conflict Groups", "Affected Rows"])
validation_findings = pd.DataFrame(findings_rows, columns=["Validation Area", "Column Name", "Issue / Observation", "Affected Rows", "Example Values / Evidence", "Suggested Action", "Finding Status"])

assert len(validation_checks) == 36, f"Expected 36 checks, got {len(validation_checks)}"
print(f"Validation checks completed: {len(validation_checks)}")
print(validation_checks["Validation Area"].value_counts().to_string())
display(validation_checks)


Validation checks completed: 36
Validation Area
Category consistency    12
Numeric                 11
ID-label mapping         8
Date                     2
Data type                2
Date chronology          1


,Validation Area,Column(s),Validation Rule,Result,Affected Rows / Count,Status,Notes
0,Numeric,Order Item Quantity,Quantity should be greater than zero,No violations detected,0,Pass,None
1,Numeric,Product Price,Product price should not be negative,No violations detected,0,Pass,None
2,Numeric,Order Item Product Price,Order item product price should not be negative,No violations detected,0,Pass,None
3,Numeric,Order Item Discount Rate,Discount rate should be between 0 and 1 as a b...,No violations detected,0,Pass,None
4,Numeric,Late_delivery_risk,Risk flag should be binary (0 or 1),No violations detected,0,Pass,None
5,Numeric,Days for shipping (real),Actual shipping duration should not be negative,No violations detected,0,Pass,None
6,Numeric,Days for shipment (scheduled),Scheduled shipping duration should not be nega...,No violations detected,0,Pass,None
7,Numeric,Latitude,Latitude must be between -90 and 90,No violations detected,0,Pass,None
8,Numeric,Longitude,Longitude must be between -180 and 180,No violations detected,0,Pass,None
9,Numeric,Order Item Discount,Discount amount should not be negative,No violations detected,0,Pass,None


In [25]:
# Step 5: Review category variants, ID-label mapping, and postal-code types
print("Category label variants (empty table means no spelling/case variants after trim):")
display(category_variants)
print("ID-label mapping review:")
display(id_label_mapping)
print("Postal-code data types:")
display(pd.DataFrame({"Column": postal_columns,
    "Current Data Type": [str(df[c].dtype) for c in postal_columns],
    "Non-empty Values": [int(df[c].notna().sum()) for c in postal_columns],
    "Unique Non-empty Values": [int(df[c].nunique(dropna=True)) for c in postal_columns]}))
print("Findings requiring review or standardization:")
display(validation_findings)


Category label variants (empty table means no spelling/case variants after trim):


,Column Name,Normalized Label,Raw Variants,Distinct Raw Variants,Affected Rows


ID-label mapping review:


,ID Column,Label Column,Mapping Direction,ID / Label,Observed Values,Conflict Groups,Affected Rows
0,Category Id,Category Name,One ID → multiple labels,,No conflicts,0,0
1,Category Id,Category Name,One label → multiple IDs,electronics,13 | 37,1,3156
2,Department Id,Department Name,One ID → multiple labels,,No conflicts,0,0
3,Product Card Id,Product Name,One ID → multiple labels,,No conflicts,0,0
4,Product Category Id,Category Name,One ID → multiple labels,,No conflicts,0,0
5,Product Category Id,Category Name,One label → multiple IDs,electronics,13 | 37,1,3156


Postal-code data types:


,Column,Current Data Type,Non-empty Values,Unique Non-empty Values
0,Customer Zipcode,float64,180516,995
1,Order Zipcode,float64,24840,609


Findings requiring review or standardization:


,Validation Area,Column Name,Issue / Observation,Affected Rows,Example Values / Evidence,Suggested Action,Finding Status
0,ID-label mapping,Category Id / Category Name,One label is associated with multiple IDs; rev...,3156,"[{'Label': 'electronics', 'IDs': ['13', '37']}]",Validate against the official category/product...,Needs review
1,ID-label mapping,Product Category Id / Category Name,One label is associated with multiple IDs; rev...,3156,"[{'Label': 'electronics', 'IDs': ['13', '37']}]",Validate against the official category/product...,Needs review
2,Data type,Customer Zipcode,Postal code is not stored as identifier text,180516,"dtype=float64; examples=['725.0', '725.0', '95...","In a cleaned copy, convert verified integer-li...",Standardization opportunity
3,Data type,Order Zipcode,Postal code is not stored as identifier text,24840,"dtype=float64; examples=['99301.0', '90049.0',...","In a cleaned copy, convert verified integer-li...",Standardization opportunity


In [26]:
# Step 6: Build the Excel report from the same results shown in this notebook
from datetime import datetime

output_path = Path("Invalid Value Analysis.xlsx")
summary = pd.DataFrame([
    {"Metric": "Dataset rows", "Result": int(df.shape[0]), "Notes": "Read from raw CSV."},
    {"Metric": "Dataset columns", "Result": int(df.shape[1]), "Notes": "Read from raw CSV."},
    {"Metric": "Exact duplicate rows", "Result": int(df.duplicated().sum()), "Notes": "Baseline metric; not included among the 36 validation checks."},
    {"Metric": "Numeric validation rules run", "Result": len(numeric_rules), "Notes": "Counts shown in Validation Checks sheet."},
    {"Metric": "Date values unparseable", "Result": int(validation_checks.loc[validation_checks["Validation Area"] == "Date", "Affected Rows / Count"].sum()), "Notes": "Across the two date columns; blank values are reported in Notes."},
    {"Metric": "Shipping dates before order dates", "Result": chronology_count, "Notes": "Chronology validation."},
    {"Metric": "Category consistency columns checked", "Result": len(category_columns), "Notes": "Whitespace-only findings were documented in the Day 1 profile."},
    {"Metric": "Validation checks run", "Result": len(validation_checks), "Notes": "11 numeric + 2 date + 1 chronology + 12 category + 8 mapping + 2 postal type checks."},
    {"Metric": "Findings requiring review", "Result": len(validation_findings), "Notes": "Includes confirmed rule violations and standardization/review opportunities; not all are confirmed errors."},
])

standardization_plan = pd.DataFrame([
    {"Area": "Text values", "Rule / Check": "Trim leading/trailing spaces", "Status from previous report": "Documented in Day 1 profile", "Recommended Action": "Trim in a cleaned copy only; preserve raw CSV.", "Status from Day 2": "Whitespace observations carried forward; not duplicated as new findings."},
    {"Area": "Category labels", "Rule / Check": "Compare after trimming whitespace and casefolding", "Status from previous report": "Checked", "Recommended Action": "Review any variants listed in Category Variants before mapping.", "Status from Day 2": "Checked"},
    {"Area": "Dates", "Rule / Check": "Parse order and shipping date/time", "Status from previous report": "Checked", "Recommended Action": "Standardize format in a working copy; keep raw values unchanged.", "Status from Day 2": "Checked"},
    {"Area": "Date chronology", "Rule / Check": "Shipping date must not precede order date", "Status from previous report": "Checked", "Recommended Action": "Investigate flagged rows; do not auto-correct.", "Status from Day 2": "Checked"},
    {"Area": "Postal codes", "Rule / Check": "Store identifiers as text", "Status from previous report": "Review recommended", "Recommended Action": "Convert carefully in a cleaned copy; do not claim leading zeros can be recovered.", "Status from Day 2": "Review recommended"},
    {"Area": "Numeric fields", "Rule / Check": "Validate against broad documented bounds", "Status from previous report": "Checked", "Recommended Action": "Confirm thresholds with the data dictionary before correcting values.", "Status from Day 2": "Checked"},
    {"Area": "ID-label mapping", "Rule / Check": "Check both ID→label and label→ID directions", "Status from previous report": "Checked", "Recommended Action": "Verify multi-ID labels against the official mapping before changing values.", "Status from Day 2": "Review any flagged mapping groups"},
])

scope_notes = pd.DataFrame([
    {"Item": "Missing-value analysis", "Day 2 handling": "Not repeated; completed by Member 2."},
    {"Item": "Whitespace-only observations", "Day 2 handling": "Documented in the Day 1 Data Profile; not counted as spelling/case variants here."},
    {"Item": "Exact duplicate rows", "Day 2 handling": "Reported as a baseline metric only; no row deletion performed."},
    {"Item": "Raw data preservation", "Day 2 handling": "All checks are read-only; the source CSV is not overwritten."},
    {"Item": "Interpretation", "Day 2 handling": "A validation flag or mapping observation is not automatically a confirmed error."},
    {"Item": "Affected records vs groups", "Day 2 handling": "Category/mapping group counts and affected-row counts are reported separately where applicable."},
])

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    validation_findings.to_excel(writer, sheet_name="Validation Findings", index=False)
    validation_checks.to_excel(writer, sheet_name="Validation Checks", index=False)
    category_variants.to_excel(writer, sheet_name="Category Variants", index=False)
    id_label_mapping.to_excel(writer, sheet_name="ID Label Mapping", index=False)
    pd.DataFrame({"Column Name": postal_columns,
                  "Current Data Type": [str(df[c].dtype) for c in postal_columns],
                  "Non-empty Unique Values": [int(df[c].nunique(dropna=True)) for c in postal_columns],
                  "Sample Values": [" | ".join(df[c].dropna().astype(str).head(8).tolist()) for c in postal_columns],
                  "Recommendation": ["Consider text storage after validating source values; preserve leading zeros where applicable." for _ in postal_columns]}).to_excel(writer, sheet_name="Postal Code Review", index=False)
    standardization_plan.to_excel(writer, sheet_name="Standardization Plan", index=False)
    scope_notes.to_excel(writer, sheet_name="Scope Notes", index=False)

print(f"Report saved: {output_path.resolve()}")
print(f"Validation rows in report: {len(validation_checks)}")
print("Source dataset modified? No.")

Report saved: /content/Invalid Value Analysis.xlsx
Validation rows in report: 36
Source dataset modified? No.


In [27]:
clean_df = raw_df.copy(deep=True)
original_shape = raw_df.shape
standardization_log = []

print("Original shape:", original_shape)
print("Working-copy shape:", clean_df.shape)
print("Raw and working copies are separate objects:", clean_df is not raw_df)

Original shape: (180519, 53)
Working-copy shape: (180519, 53)
Raw and working copies are separate objects: True


In [28]:
#Standardize whitespace in text columns

text_columns = clean_df.select_dtypes(include=["object", "string"]).columns.tolist()

for column in text_columns:
    before = clean_df[column].copy()
    # pandas StringDtype keeps missing values as <NA>, not the literal text "nan".
    after = before.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    changed_mask = before.astype("string").fillna("<MISSING>") != after.fillna("<MISSING>")
    changed_rows = int(changed_mask.sum())
    if changed_rows:
        standardization_log.append({
            "Column": column,
            "Transformation": "Trim leading/trailing whitespace and collapse repeated internal spaces",
            "Rows affected": changed_rows,
            "Before dtype": str(before.dtype),
            "After dtype": str(after.dtype),
            "Notes": "Text normalization only; no spelling or ID-label mapping changes made."
        })
    clean_df[column] = after

print("Text columns checked:", len(text_columns))
print("Columns with whitespace changes:", sum(1 for x in standardization_log if "whitespace" in x["Transformation"].lower()))
print("Rows changed are recorded per column in standardization_log.")

Text columns checked: 24
Columns with whitespace changes: 6
Rows changed are recorded per column in standardization_log.


In [29]:
#Convert order and shipping date columns to datetime

from datetime import datetime

date_columns = ["order date (DateOrders)", "shipping date (DateOrders)"]
for column in date_columns:
    if column not in clean_df.columns:
        print(f"WARNING: Missing expected date column: {column}")
        continue
    before = clean_df[column].copy()
    parsed = pd.to_datetime(before, errors="coerce")
    # Count only non-empty original values that failed parsing.
    nonempty_before = before.astype("string").str.strip().ne("") & before.notna()
    newly_invalid = int((nonempty_before & parsed.isna()).sum())
    clean_df[column] = parsed
    standardization_log.append({
        "Column": column,
        "Transformation": "Parse as datetime",
        "Rows affected": int(before.astype("string").fillna("<MISSING>").ne(parsed.astype("string").fillna("<MISSING>")).sum()),
        "Before dtype": str(before.dtype),
        "After dtype": str(clean_df[column].dtype),
        "Notes": f"Newly unparseable non-empty values: {newly_invalid}; no rows dropped."
    })
    print(f"{column}: dtype={clean_df[column].dtype}; newly unparseable non-empty values={newly_invalid}")



order date (DateOrders): dtype=datetime64[ns]; newly unparseable non-empty values=0
shipping date (DateOrders): dtype=datetime64[ns]; newly unparseable non-empty values=0


In [30]:
# Convert identifier columns to string

id_columns = [
    "Order Id", "Order Item Id", "Customer Id", "Order Customer Id",
    "Product Card Id", "Order Item Cardprod Id", "Category Id",
    "Department Id", "Product Category Id"
]

for column in id_columns:
    if column not in clean_df.columns:
        continue
    before = clean_df[column].copy()
    after = before.astype("Int64").astype("string") if pd.api.types.is_integer_dtype(before.dtype) else before.astype("string")
    clean_df[column] = after
    standardization_log.append({
        "Column": column,
        "Transformation": "Store identifier as string",
        "Rows affected": int(before.astype("string").fillna("<MISSING>").ne(after.fillna("<MISSING>")).sum()),
        "Before dtype": str(before.dtype),
        "After dtype": str(after.dtype),
        "Notes": "Values preserved as identifiers; no IDs remapped."
    })

print("ID columns processed:", [c for c in id_columns if c in clean_df.columns])

ID columns processed: ['Order Id', 'Order Item Id', 'Customer Id', 'Order Customer Id', 'Product Card Id', 'Order Item Cardprod Id', 'Category Id', 'Department Id', 'Product Category Id']


In [31]:
# Leading zeros cannot be reconstructed

postal_columns = ["Customer Zipcode", "Order Zipcode"]

def postal_to_text(value):
    if pd.isna(value):
        return pd.NA
    text_value = str(value).strip()
    # Convert only values that are numeric and integer-like; otherwise preserve text.
    try:
        numeric_value = float(text_value)
        if np.isfinite(numeric_value) and numeric_value.is_integer():
            return str(int(numeric_value))
    except (TypeError, ValueError):
        pass
    return text_value

for column in postal_columns:
    if column not in clean_df.columns:
        continue
    before = clean_df[column].copy()
    after = before.map(postal_to_text).astype("string")
    changed_rows = int(before.astype("string").fillna("<MISSING>").ne(after.fillna("<MISSING>")).sum())
    clean_df[column] = after
    standardization_log.append({
        "Column": column,
        "Transformation": "Store postal code as text; remove .0 only from integer-like values",
        "Rows affected": changed_rows,
        "Before dtype": str(before.dtype),
        "After dtype": str(after.dtype),
        "Notes": "Missing values preserved. Leading zeros cannot be recovered unless present in source data."
    })
    print(f"{column}: {before.dtype} -> {after.dtype}; rows whose representation changed={changed_rows}; missing={int(after.isna().sum())}")

Customer Zipcode: float64 -> string; rows whose representation changed=180516; missing=3
Order Zipcode: float64 -> string; rows whose representation changed=24840; missing=155679


In [32]:

review_columns = [c for c in [
    "Category Name", "Department Name", "Order Region", "Delivery Status",
    "Type", "Customer Country", "Order Country", "Customer Segment",
    "Market", "Order Status", "Shipping Mode", "Product Name"
] if c in clean_df.columns]

for column in review_columns:
    print(f"\n{column}: {clean_df[column].nunique(dropna=True)} unique non-missing values")
    print(clean_df[column].dropna().astype(str).value_counts().head(5).to_dict())

for column in date_columns:
    if column in clean_df.columns:
        print(f"\n{column} sample:")
        print(clean_df[column].head(3).dt.strftime("%Y-%m-%d %H:%M:%S").tolist())

print("\nImportant: Electronics mapping observation (Category Name linked to IDs 13 and 37) is not auto-corrected.")


Category Name: 50 unique non-missing values
{'Cleats': 24551, "Men's Footwear": 22246, "Women's Apparel": 21035, 'Indoor/Outdoor Games': 19298, 'Fishing': 17325}

Department Name: 11 unique non-missing values
{'Fan Shop': 66861, 'Apparel': 48998, 'Golf': 33220, 'Footwear': 14525, 'Outdoors': 9686}

Order Region: 23 unique non-missing values
{'Central America': 28341, 'Western Europe': 27109, 'South America': 14935, 'Oceania': 10148, 'Northern Europe': 9792}

Delivery Status: 4 unique non-missing values
{'Late delivery': 98977, 'Advance shipping': 41592, 'Shipping on time': 32196, 'Shipping canceled': 7754}

Type: 4 unique non-missing values
{'DEBIT': 69295, 'TRANSFER': 49883, 'PAYMENT': 41725, 'CASH': 19616}

Customer Country: 2 unique non-missing values
{'EE. UU.': 111146, 'Puerto Rico': 69373}

Order Country: 164 unique non-missing values
{'Estados Unidos': 24840, 'Francia': 13222, 'México': 13172, 'Alemania': 9564, 'Australia': 8497}

Customer Segment: 3 unique non-missing values
{

In [33]:
# Build cleaning documentation and change log

cleaning_documentation = pd.DataFrame([
    {"Area": "Raw data preservation", "Change made": "Created clean_df as a deep copy of raw_df", "Reason": "Keep the source dataset unchanged", "Caveat": "Raw CSV is not overwritten"},
    {"Area": "Text fields", "Change made": "Trimmed leading/trailing whitespace and collapsed repeated internal spaces", "Reason": "Normalize text formatting", "Caveat": "No spelling corrections or automatic case changes"},
    {"Area": "Date fields", "Change made": "Parsed order and shipping dates to datetime", "Reason": "Enable reliable date/time operations", "Caveat": "Unparseable non-empty values are counted; rows are not dropped"},
    {"Area": "ID fields", "Change made": "Converted selected identifier columns to string", "Reason": "Identifiers are not measurements", "Caveat": "No ID values remapped"},
    {"Area": "Postal codes", "Change made": "Converted integer-like postal code values to text without trailing .0", "Reason": "Postal codes are identifiers", "Caveat": "Missing values retained; lost leading zeros cannot be recovered"},
    {"Area": "Category mappings", "Change made": "Reviewed category labels; did not automatically resolve multi-ID labels", "Reason": "Avoid unsupported data corrections", "Caveat": "Verify mapping against an authoritative data dictionary"},
    {"Area": "Row retention", "Change made": "No rows dropped", "Reason": "Interim standardization only", "Caveat": "Invalid-value remediation is a separate decision"}
])

change_log_df = pd.DataFrame(standardization_log)
print("Documentation rows:", len(cleaning_documentation))
print("Transformation log rows:", len(change_log_df))
display(cleaning_documentation)
display(change_log_df)

Documentation rows: 7
Transformation log rows: 19


,Area,Change made,Reason,Caveat
0,Raw data preservation,Created clean_df as a deep copy of raw_df,Keep the source dataset unchanged,Raw CSV is not overwritten
1,Text fields,Trimmed leading/trailing whitespace and collap...,Normalize text formatting,No spelling corrections or automatic case changes
2,Date fields,Parsed order and shipping dates to datetime,Enable reliable date/time operations,Unparseable non-empty values are counted; rows...
3,ID fields,Converted selected identifier columns to string,Identifiers are not measurements,No ID values remapped
4,Postal codes,Converted integer-like postal code values to t...,Postal codes are identifiers,Missing values retained; lost leading zeros ca...
5,Category mappings,Reviewed category labels; did not automaticall...,Avoid unsupported data corrections,Verify mapping against an authoritative data d...
6,Row retention,No rows dropped,Interim standardization only,Invalid-value remediation is a separate decision


,Column,Transformation,Rows affected,Before dtype,After dtype,Notes
0,Category Name,Trim leading/trailing whitespace and collapse ...,1543,object,string,Text normalization only; no spelling or ID-lab...
1,Customer Street,Trim leading/trailing whitespace and collapse ...,3036,object,string,Text normalization only; no spelling or ID-lab...
2,Department Name,Trim leading/trailing whitespace and collapse ...,362,object,string,Text normalization only; no spelling or ID-lab...
3,Order Region,Trim leading/trailing whitespace and collapse ...,17925,object,string,Text normalization only; no spelling or ID-lab...
4,Product Image,Trim leading/trailing whitespace and collapse ...,886,object,string,Text normalization only; no spelling or ID-lab...
5,Product Name,Trim leading/trailing whitespace and collapse ...,1774,object,string,Text normalization only; no spelling or ID-lab...
6,order date (DateOrders),Parse as datetime,180519,string,datetime64[ns],Newly unparseable non-empty values: 0; no rows...
7,shipping date (DateOrders),Parse as datetime,180519,string,datetime64[ns],Newly unparseable non-empty values: 0; no rows...
8,Order Id,Store identifier as string,0,int64,string,Values preserved as identifiers; no IDs remapped.
9,Order Item Id,Store identifier as string,0,int64,string,Values preserved as identifiers; no IDs remapped.


In [34]:
output_csv = Path("DataCo_Cleaned_Interim.csv")
documentation_xlsx = Path("Cleaning_Documentation.xlsx")
log_xlsx = Path("Standardization_Log.xlsx")

# CSV date strings use a consistent 24-hour format.
output_df = clean_df.copy()
for column in date_columns:
    if column in output_df.columns and pd.api.types.is_datetime64_any_dtype(output_df[column]):
        output_df[column] = output_df[column].dt.strftime("%Y-%m-%d %H:%M:%S")

output_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
with pd.ExcelWriter(documentation_xlsx, engine="openpyxl") as writer:
    cleaning_documentation.to_excel(writer, sheet_name="Cleaning Documentation", index=False)
    pd.DataFrame([{"Metric": "Original rows", "Value": original_shape[0]},
                  {"Metric": "Cleaned rows", "Value": clean_df.shape[0]},
                  {"Metric": "Original columns", "Value": original_shape[1]},
                  {"Metric": "Cleaned columns", "Value": clean_df.shape[1]},
                  {"Metric": "Rows dropped", "Value": original_shape[0] - clean_df.shape[0]}]).to_excel(writer, sheet_name="Verification", index=False)
with pd.ExcelWriter(log_xlsx, engine="openpyxl") as writer:
    change_log_df.to_excel(writer, sheet_name="Standardization Log", index=False)

print("Saved:", output_csv.resolve())
print("Saved:", documentation_xlsx.resolve())
print("Saved:", log_xlsx.resolve())

Saved: /content/DataCo_Cleaned_Interim.csv
Saved: /content/Cleaning_Documentation.xlsx
Saved: /content/Standardization_Log.xlsx


In [35]:


assert clean_df.shape == raw_df.shape, "Row/column count changed unexpectedly."
assert len(clean_df) == len(raw_df), "Rows were added or removed unexpectedly."
assert raw_df.equals(df), "The original in-memory raw_df reference differs from df."

print("Original shape:", raw_df.shape)
print("Cleaned shape:", clean_df.shape)
print("Rows dropped:", len(raw_df) - len(clean_df))
print("Original raw dataframe unchanged:", raw_df.equals(df))
print("Output files exist:", output_csv.exists(), documentation_xlsx.exists(), log_xlsx.exists())
display(clean_df.head())

Original shape: (180519, 53)
Cleaned shape: (180519, 53)
Rows dropped: 0
Original raw dataframe unchanged: True
Output files exist: True True True


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,<NA>,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,<NA>,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,<NA>,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,<NA>,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,<NA>,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class
